# Librerias

In [1]:
!pip install pymongo

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [2]:
import pymongo
from pymongo import MongoClient, GEOSPHERE
from pprint import pprint
import json
import os
import requests


# Script solo para descarga de datos

In [3]:
# Directorio donde guardaremos los datos
os.makedirs("data/raw", exist_ok=True)

# URLs directas a los portales de Datos Abiertos (URLs estables a fecha de hoy)
urls = {
    "alojamientos.json": "https://datos.comunidad.madrid/dataset/ef437c50-5b26-4843-92c3-d8eef5507fac/resource/20237d3a-718f-4179-9631-fc2f4a9fe566/download/alojamientos_turisticos.json",
    "museos.json": "https://datos.madrid.es/egob/catalogo/201132-0-museos.json",
    "actividades.json": "https://datos.madrid.es/egob/catalogo/206974-0-agenda-eventos-culturales-100.json"
}

print("--- Iniciando descarga de Datasets ---")

for nombre_archivo, url in urls.items():
    print(f"Descargando {nombre_archivo}...")
    try:
        response = requests.get(url)
        response.raise_for_status() # Lanza error si la web falla (404, 500)

        # Guardamos el fichero en la carpeta data/raw
        ruta_completa = os.path.join("data/raw", nombre_archivo)
        with open(ruta_completa, 'wb') as f:
            f.write(response.content)

        print(f"✅ {nombre_archivo} guardado correctamente ({len(response.content)/1024:.2f} KB).")

    except Exception as e:
        print(f"❌ Error descargando {nombre_archivo}: {e}")

print("\n--- Descarga completada. Listos para cargar en MongoDB ---")

--- Iniciando descarga de Datasets ---
Descargando alojamientos.json...
✅ alojamientos.json guardado correctamente (4397.07 KB).
Descargando museos.json...
✅ museos.json guardado correctamente (128.26 KB).
Descargando actividades.json...
✅ actividades.json guardado correctamente (1838.35 KB).

--- Descarga completada. Listos para cargar en MongoDB ---


# Carga de Datos a MongoDB

Desde un archivo .bat <br>
cd D:\App\mongodb\bin <br>
mongoimport --verbose --db Proyecto --collection accommodations --file "D:\MaestriaUNIR\1. Bases de datos para el Big Data\Actividad2\datos\alojamientos.json" --jsonArray <br>
mongoimport --verbose --db Proyecto --collection cultural_pois --file "D:\MaestriaUNIR\1. Bases de datos para el Big Data\Actividad2\datos\museos.json" --jsonArray <br>
mongoimport --verbose --db Proyecto --collection tourist_events --file "D:\MaestriaUNIR\1. Bases de datos para el Big Data\Actividad2\datos\actividades.json" --jsonArray <br>


In [5]:

# 1. CONFIGURACIÓN
# ---------------------------------------------------------
# Ajusta esta ruta a la carpeta donde descargaste tus archivos
RUTA_ARCHIVOS = "data/raw/"
DB_NAME = "Proyecto"

# Diccionario que mapea: 'Nombre del archivo' -> 'Nombre de la colección destino'
archivos_colecciones = {
    "alojamientos.json": "alojamientos",
    "museos.json": "museos",
    "actividades.json": "actividades"
}

# 2. CONEXIÓN A MONGODB
# ---------------------------------------------------------
try:
    client = MongoClient('mongodb://localhost:27017/')
    db = client[DB_NAME]
    print(f"✅ Conectado exitosamente a la base de datos: {DB_NAME}")
except Exception as e:
    print(f"❌ Error de conexión: {e}")
    exit()

# 3. FUNCIÓN DE CARGA
# ---------------------------------------------------------
def cargar_json_a_mongo(nombre_archivo, nombre_coleccion):
    ruta_completa = os.path.join(RUTA_ARCHIVOS, nombre_archivo)

    if not os.path.exists(ruta_completa):
        print(f"⚠️ El archivo {nombre_archivo} no existe en {RUTA_ARCHIVOS}. Saltando...")
        return

    print(f"📂 Leyendo archivo: {nombre_archivo}...")

    try:
        with open(ruta_completa, 'r', encoding='utf-8') as file:
            # CORRECCIÓN AQUÍ:
            # Añadimos strict=False para permitir caracteres de control (saltos de línea) dentro de los textos.
            datos = json.load(file, strict=False)

        # Validación: MongoDB espera una lista de diccionarios o un diccionario único
        if isinstance(datos, list):
            if len(datos) > 0:
                result = db[nombre_coleccion].insert_many(datos)
                print(f"   🚀 Insertados {len(result.inserted_ids)} documentos en '{nombre_coleccion}'.")
            else:
                print(f"   ⚠️ El archivo {nombre_archivo} está vacío.")

        elif isinstance(datos, dict):
            db[nombre_coleccion].insert_one(datos)
            print(f"   🚀 Insertado 1 documento en '{nombre_coleccion}'.")

        else:
            print(f"   ❌ Formato JSON no reconocido en {nombre_archivo}.")

    except json.JSONDecodeError as e:
        # Si falla incluso con strict=False, intentamos una limpieza manual más agresiva
        print(f"   ⚠️ Error de formato JSON simple. Intentando limpieza agresiva para {nombre_archivo}...")
        try:
            with open(ruta_completa, 'r', encoding='utf-8') as file:
                raw_data = file.read()
                # Eliminamos caracteres de control molestos manualmente
                clean_data = "".join(ch for ch in raw_data if (ord(ch) >= 32 or ch == '\n' or ch == '\r' or ch == '\t'))
                datos = json.loads(clean_data, strict=False)

                if isinstance(datos, list) and len(datos) > 0:
                     result = db[nombre_coleccion].insert_many(datos)
                     print(f"   🚀 ¡Recuperado! Insertados {len(result.inserted_ids)} documentos tras limpieza agresiva.")
        except Exception as e2:
             print(f"   ❌ Falló la recuperación manual: {e2}")

    except Exception as e:
        print(f"   ❌ Error general procesando {nombre_archivo}: {e}")
        # 4. EJECUCIÓN DEL PROCESO
# ---------------------------------------------------------
print("\n--- Iniciando Ingesta de Datos ---\n")

# Limpiamos las colecciones antes de cargar para evitar duplicados en pruebas
# (Comenta estas líneas si quieres acumular datos)
for _, coleccion in archivos_colecciones.items():
    db[coleccion].drop()
    print(f"🧹 Colección '{coleccion}' limpiada.")

# Bucle principal de carga
for archivo, coleccion in archivos_colecciones.items():
    cargar_json_a_mongo(archivo, coleccion)

print("\n--- Proceso Finalizado ---")
print("Ahora puedes ejecutar el Pipeline de Agregación para transformar los datos.")

✅ Conectado exitosamente a la base de datos: Proyecto

--- Iniciando Ingesta de Datos ---

🧹 Colección 'alojamientos' limpiada.
🧹 Colección 'museos' limpiada.
🧹 Colección 'actividades' limpiada.
📂 Leyendo archivo: alojamientos.json...
   🚀 Insertado 1 documento en 'alojamientos'.
📂 Leyendo archivo: museos.json...
   🚀 Insertado 1 documento en 'museos'.
📂 Leyendo archivo: actividades.json...
   🚀 Insertado 1 documento en 'actividades'.

--- Proceso Finalizado ---
Ahora puedes ejecutar el Pipeline de Agregación para transformar los datos.


# Pipeline de Agregación para transformar los datos

In [ ]:
import json
import os
from pymongo import MongoClient

# --- CONFIGURACIÓN ---
client = MongoClient('mongodb://localhost:27017/')
db = client['Proyecto']
RUTA_ARCHIVOS = "data/raw/" 
# MANTENEMOS TUS NOMBRES EN ESPAÑOL PARA LA CARGA
archivos_colecciones = {
    "alojamientos.json": "alojamientos",
    "museos.json": "museos",
    "actividades.json": "actividades"
}

print("--- 🚀 INICIANDO SCRIPT MAESTRO (CORREGIDO) ---")

# 1. CARGA (Esta parte ya te funcionaba bien, la dejo igual)
# ---------------------------------------------------------
for archivo, coleccion in archivos_colecciones.items():
    print(f"\n📂 Procesando {archivo} -> Colección '{coleccion}'...")
    db[coleccion].drop() 

    ruta = os.path.join(RUTA_ARCHIVOS, archivo)
    if not os.path.exists(ruta):
        print(f"   ⚠️ ARCHIVO NO ENCONTRADO: {ruta}")
        continue

    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            raw_data = json.load(f, strict=False)
        
        data_to_insert = []

        if isinstance(raw_data, list):
            data_to_insert = raw_data
        elif isinstance(raw_data, dict):
            for key in ['data', '@graph', 'graph', 'result']:
                if key in raw_data and isinstance(raw_data[key], list):
                    data_to_insert = raw_data[key]
                    print(f"   📦 Datos desempaquetados de la clave '{key}'")
                    break
            if not data_to_insert:
                data_to_insert = [raw_data]

        if data_to_insert:
            db[coleccion].insert_many(data_to_insert)
            print(f"   ✅ Cargados {len(data_to_insert)} documentos crudos.")
        else:
            print("   ⚠️ No se encontraron datos válidos.")

    except Exception as e:
        print(f"   ❌ Error cargando {archivo}: {e}")


# 2. TRANSFORMACIÓN (AQUÍ ESTABA EL ERROR)
# ---------------------------------------------------------
print("\n⚙️ Ejecutando Pipelines de Transformación (ETL)...")

# A. ALOJAMIENTOS 
# CORRECCIÓN: Leemos de db.alojamientos (español)
pipeline_hotel = [
    {
        "$project": {
            "name": { "$ifNull": ["$denominacion", "$nombre", "Sin Nombre"] },
            "type": "Hotel",
            "category": "$categoria",
            "address": {
                "street": "$via_nombre",
                "number": "$numero",
                "zip_code": "$cdpostal", 
                "locality": "$localidad"
            },
            "location": None
        }
    },
    { "$out": "alojamientos_limpio" }
]
try:
    # OJO: Aquí cambiamos db.alojamientos por db.alojamientos_limpio
    db.alojamientos.aggregate(pipeline_hotel) 
    c = db.alojamientos_limpio.count_documents({})
    print(f"   🏨 Alojamientos limpios: {c}")
except Exception as e: print(f"Error Hotel: {e}")


# B. MUSEOS
# CORRECCIÓN: Leemos de db.museos
pipeline_museos = [
    {
        "$project": {
            "name": { "$ifNull": ["$title", "$nombre"] },
            "category": "Museo",
            "contact": "$relation",
            "location": {
                "type": "Point",
                "coordinates": [
                    { "$convert": { "input": "$location.longitude", "to": "double", "onError": None, "onNull": None } }, 
                    { "$convert": { "input": "$location.latitude", "to": "double", "onError": None, "onNull": None } }
                ]
            }
        }
    },
    { "$match": { "location.coordinates.0": { "$type": "number" } } },
    { "$out": "museos_limpio" }
]
try:
    # OJO: Aquí cambiamos db.museos por db.museos_limpio
    db.museos.aggregate(pipeline_museos)
    c = db.museos_limpio.count_documents({})
    print(f"   🏛️ Museos geolocalizados: {c}")
except Exception as e: print(f"Error Museo: {e}")


# C. ACTIVIDADES
# CORRECCIÓN: Leemos de db.actividades
pipeline_eventos = [
    {
        "$project": {
            "title": "$title",
            "category": "Evento",
            "date_info": { "start": "$dtstart", "end": "$dtend" },
            "location": {
                "type": "Point",
                "coordinates": [
                    { "$convert": { "input": "$location.longitude", "to": "double", "onError": None, "onNull": None } }, 
                    { "$convert": { "input": "$location.latitude", "to": "double", "onError": None, "onNull": None } }
                ]
            }
        }
    },
    { "$match": { "location.coordinates.0": { "$type": "number" } } },
    { "$out": "actividades_limpio" }
]
try:
    # OJO: Aquí cambiamos db.actividades por db.actividades_limpio
    db.actividades.aggregate(pipeline_eventos)
    c = db.actividades_limpio.count_documents({})
    print(f"   🎭 Actividades geolocalizadas: {c}")
except Exception as e: print(f"Error Eventos: {e}")


# 3. ÍNDICES
# ---------------------------------------------------------
print("\n🗺️ Creando índices espaciales...")
if db.cultural_pois_clean.count_documents({}) > 0:
    db.cultural_pois_clean.create_index([("location", "2dsphere")])
if db.tourist_events_clean.count_documents({}) > 0:
    db.tourist_events_clean.create_index([("location", "2dsphere")])

print("\n✅ --- PROCESO COMPLETADO ---")

--- 🚀 INICIANDO SCRIPT MAESTRO (CORREGIDO) ---

📂 Procesando alojamientos.json -> Colección 'alojamientos'...
   📦 Datos desempaquetados de la clave 'data'
   ✅ Cargados 10768 documentos crudos.

📂 Procesando museos.json -> Colección 'museos'...
   📦 Datos desempaquetados de la clave '@graph'
   ✅ Cargados 69 documentos crudos.

📂 Procesando actividades.json -> Colección 'actividades'...
   📦 Datos desempaquetados de la clave '@graph'
   ✅ Cargados 944 documentos crudos.

⚙️ Ejecutando Pipelines de Transformación (ETL)...
   🏨 Alojamientos limpios: 10768
   🏛️ Museos geolocalizados: 69
   🎭 Actividades geolocalizadas: 859

🗺️ Creando índices espaciales...

✅ --- PROCESO COMPLETADO ---


### Consulta de Inteligencia de Negocio (BI)
- Contar cuántos eventos culturales hay esta semana a menos de 1km de los principales museos, para crear paquetes turísticos conjuntos.

In [ ]:
from pymongo import MongoClient
from datetime import datetime, timedelta
from dateutil import parser # Librería estándar en Anaconda/Colab para leer fechas ISO

client = MongoClient('mongodb://localhost:27017/')
db = client['Proyecto']

print("--- 💼 GENERANDO PAQUETES TURÍSTICOS (MUSEO + EVENTO) ---")

# 1. DEFINICIÓN DEL FILTRO TEMPORAL ("ESTA SEMANA")
# -----------------------------------------------------
# NOTA: Si tus datos son antiguos (ej. 2023), cambia 'HOY' por una fecha fija
# para simular que estás en el pasado. Ej: datetime(2024, 5, 20)
HOY = datetime.now()
SEMANA_QUE_VIENE = HOY + timedelta(days=7)

print(f"📅 Buscando eventos entre: {HOY.date()} y {SEMANA_QUE_VIENE.date()}")

# 2. CONFIGURACIÓN ESPACIAL (1 KM)
RADIO_KM = 1.0
RADIO_RADIANES = RADIO_KM / 6378.1

# 3. OBTENER MUSEOS
museos = list(db.cultural_pois_clean.find({}, {"name": 1, "location": 1}))
paquetes_turisticos = []

print(f"🔄 Cruzando {len(museos)} museos con la agenda cultural...")

for museo in museos:
    if "location" not in museo or not museo["location"]:
        continue

    # A. BUSQUEDA ESPACIAL (Traemos TODO lo que esté cerca, sin mirar fecha aún)
    query_espacial = {
        "location": {
            "$geoWithin": {
                "$centerSphere": [museo["location"]["coordinates"], RADIO_RADIANES]
            }
        }
    }

    # Traemos título y fecha de los eventos cercanos
    eventos_cercanos = list(db.tourist_events_clean.find(
        query_espacial,
        {"title": 1, "date_info": 1}
    ))

    # B. FILTRO TEMPORAL (Procesamos en Python para precisión)
    eventos_validos = []

    for evento in eventos_cercanos:
        try:
            # Obtenemos la fecha de inicio del evento
            # Adaptamos según tu esquema: a veces es 'start', 'dtstart', o string directo
            fecha_str = evento.get("date_info", {}).get("start")

            if fecha_str:
                # Parseamos la fecha (funciona con ISO, "2025-05-10", etc.)
                fecha_obj = parser.parse(fecha_str, fuzzy=True).replace(tzinfo=None)

                # LA LÓGICA DE NEGOCIO: ¿Cae en esta semana?
                if HOY <= fecha_obj <= SEMANA_QUE_VIENE:
                    eventos_validos.append({
                        "titulo": evento.get("title"),
                        "fecha": fecha_obj.strftime("%d/%m/%Y"),
                        "dia": fecha_obj.strftime("%A") # Día de la semana
                    })
        except Exception:
            # Si la fecha está mal formada, ignoramos el evento silenciosamente
            continue

    # C. SI HAY MATCH, CREAMOS EL PAQUETE
    if len(eventos_validos) > 0:
        paquetes_turisticos.append({
            "Museo_Cabecera": museo.get("name"),
            "Cantidad_Opciones": len(eventos_validos),
            "Mejor_Opcion": eventos_validos[0] # Sugerimos la primera
        })

# 4. ORDENAR Y MOSTRAR RESULTADOS
paquetes_turisticos.sort(key=lambda x: x["Cantidad_Opciones"], reverse=True)

print(f"\n✅ Se han generado {len(paquetes_turisticos)} posibles paquetes combinados.")
print("=" * 60)

if len(paquetes_turisticos) == 0:
    print("⚠️ AVISO: No se encontraron eventos PARA ESTA SEMANA.")
    print("   Posible causa: El dataset descargado tiene datos de meses pasados.")
    print("   Solución: Cambia la variable 'HOY' en el script a una fecha pasada (ej. 2024).")
else:
    for pkg in paquetes_turisticos[:5]: # Top 5
        print(f"\n🏛️ PACK CULTURAL: {pkg['Museo_Cabecera']}")
        print(f"   🔥 {pkg['Cantidad_Opciones']} eventos disponibles a <1km esta semana.")
        print(f"   💡 Sugerencia: Visita al museo + '{pkg['Mejor_Opcion']['titulo']}'")
        print(f"      Cuándo: {pkg['Mejor_Opcion']['dia']} ({pkg['Mejor_Opcion']['fecha']})")

--- 💼 GENERANDO PAQUETES TURÍSTICOS (MUSEO + EVENTO) ---
📅 Buscando eventos entre: 2026-01-11 y 2026-01-18
🔄 Cruzando 69 museos con la agenda cultural...

✅ Se han generado 58 posibles paquetes combinados.

🏛️ PACK CULTURAL: Museo Nacional de Artes Decorativas
   🔥 20 eventos disponibles a <1km esta semana.
   💡 Sugerencia: Visita al museo + 'Artes en el Retiro'
      Cuándo: Saturday (17/01/2026)

🏛️ PACK CULTURAL: Casón del Buen Retiro
   🔥 19 eventos disponibles a <1km esta semana.
   💡 Sugerencia: Visita al museo + 'Artes en el Retiro'
      Cuándo: Saturday (17/01/2026)

🏛️ PACK CULTURAL: Museo de Cera
   🔥 18 eventos disponibles a <1km esta semana.
   💡 Sugerencia: Visita al museo + 'Actividades en Iglesia de San Antón. 17 enero'
      Cuándo: Saturday (17/01/2026)

🏛️ PACK CULTURAL: Museo del Seguro. Fundación Mapfre
   🔥 18 eventos disponibles a <1km esta semana.
   💡 Sugerencia: Visita al museo + 'Actividades en Iglesia de San Antón. 17 enero'
      Cuándo: Saturday (17/01/202

"Identificación de 'Puntos Calientes' Culturales (Hotspots): Como analista de turismo, quiero saber cuáles son los museos de Madrid que tienen una mayor oferta de eventos y actividades a su alrededor (en un radio de 1 km) para crear 'Packs Turísticos Combinados' (Visita al Museo + Actividad cercana)."

In [ ]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['Proyecto']

print("--- 📊 INICIANDO ANÁLISIS DE PUNTOS CALIENTES (VERSIÓN ROBUSTA) ---")

# 1. Configuración Matemática
# $geoWithin usa "Radianes" para cálculos en la esfera terrestre.
# Fórmula: km / radio_tierra_km
RADIO_KM = 1.0
RADIO_TIERRA_KM = 6378.1
RADIO_RADIANES = RADIO_KM / RADIO_TIERRA_KM

# 2. Obtenemos museos
museos = list(db.cultural_pois_clean.find({}, {"name": 1, "location": 1}))
resultados_ranking = []

print(f"🔄 Analizando entorno de {len(museos)} museos...")

# 3. Bucle de Análisis
for museo in museos:
    # Verificación de seguridad: si el museo no tiene coordenadas, saltamos
    if "location" not in museo or not museo["location"]:
        continue

    coords_museo = museo["location"]["coordinates"]

    # --- CAMBIO IMPORTANTE AQUÍ ---
    # Usamos $geoWithin con $centerSphere en lugar de $near.
    # Sintaxis: [ [Longitud, Latitud], Radio_en_Radianes ]
    query_area = {
        "location": {
            "$geoWithin": {
                "$centerSphere": [ coords_museo, RADIO_RADIANES ]
            }
        }
    }

    # Ahora count_documents no fallará
    try:
        total_eventos = db.tourist_events_clean.count_documents(query_area)

        # Recuperar ejemplos (opcional)
        ejemplos = []
        if total_eventos > 0:
            cursor = db.tourist_events_clean.find(query_area, {"title": 1}).limit(3)
            ejemplos = [doc.get("title", "Sin título") for doc in cursor]

        resultados_ranking.append({
            "Museo": museo.get("name", "Desconocido"),
            "Total_Eventos": total_eventos,
            "Ejemplos": ejemplos
        })
    except Exception as e:
        print(f"   ⚠️ Error procesando {museo.get('name')}: {e}")

# 4. Ordenar y Mostrar
resultados_ranking.sort(key=lambda x: x["Total_Eventos"], reverse=True)

print(f"\n🏆 TOP 5 MUSEOS CON MÁS VIDA CULTURAL (Radio: {RADIO_KM} km):")
print("=" * 60)

for i, item in enumerate(resultados_ranking[:5], 1):
    print(f"\n#{i}. {item['Museo']}")
    print(f"   🔥 Densidad de Eventos: {item['Total_Eventos']}")
    if item['Ejemplos']:
        print(f"   👀 Ejemplos: {item['Ejemplos']}")

print("\n✅ Análisis finalizado.")

--- 📊 INICIANDO ANÁLISIS DE PUNTOS CALIENTES (VERSIÓN ROBUSTA) ---
🔄 Analizando entorno de 69 museos...

🏆 TOP 5 MUSEOS CON MÁS VIDA CULTURAL (Radio: 1.0 km):

#1. Museo Nacional de Artes Decorativas
   🔥 Densidad de Eventos: 109
   👀 Ejemplos: ['Árboles de El Retiro', 'Árboles de El Retiro', 'Artes en el Retiro']

#2. Casón del Buen Retiro
   🔥 Densidad de Eventos: 99
   👀 Ejemplos: ['Árboles de El Retiro', 'Árboles de El Retiro', 'Artes en el Retiro']

#3. Real Monasterio de Santa Isabel
   🔥 Densidad de Eventos: 98
   👀 Ejemplos: ['Árboles de El Retiro', 'Árboles de El Retiro', 'Artes en el Retiro']

#4. Museo Nacional de Antropología
   🔥 Densidad de Eventos: 94
   👀 Ejemplos: ["Ruta 'Los caprichos del Retiro'", "Ruta 'Los orígenes del Retiro'", "Senda 'El invierno en el Retiro'"]

#5. La Casa Encendida de la Fundación Montemadrid
   🔥 Densidad de Eventos: 93
   👀 Ejemplos: ['Los paseantes: rutas para redescubrir Madrid', 'Madrid, Musa de las Letras', 'Metamorfosis: espacio y socie

# Documentacion

In [ ]:
!pip install python-docx

In [ ]:
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

# Crear el documento
doc = Document()

# Título Principal
titulo = doc.add_heading('Diseño del Esquema de Datos NoSQL - Proyecto DataTA', 0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

# --- SECCIÓN 1: DISEÑO DEL ESQUEMA ---
doc.add_heading('1. Diseño del Esquema de Datos NoSQL (MongoDB)', level=1)
doc.add_paragraph(
    "Para este proyecto, se ha diseñado una arquitectura basada en Colecciones Desacopladas "
    "con Documentos Embebidos. A continuación se detalla el diseño JSON para las tres colecciones resultantes del proceso ETL."
)

# Función auxiliar para añadir código JSON con formato
def add_json_block(document, json_text, title):
    p_title = document.add_paragraph()
    runner = p_title.add_run(title)
    runner.bold = True
    runner.font.size = Pt(11)

    p_code = document.add_paragraph(json_text)
    p_code.style = 'No Spacing'
    for run in p_code.runs:
        run.font.name = 'Courier New'
        run.font.size = Pt(9)
        run.font.color.rgb = RGBColor(0, 50, 100)
    document.add_paragraph() # Espacio

# JSON A: Alojamientos
json_hotel = """{
  "_id": "ObjectId('...')",
  "name": "String (Ej: Gran Hotel Reina Victoria)",
  "type": "String (Ej: Hotel, Pensión, Apartamento)",
  "category": "String (Ej: 4 Estrellas, Lujo)",
  "address": {  // <--- PATRÓN EMBEBIDO (Agrupación lógica)
    "street": "String (Nombre de vía)",
    "number": "String",
    "zip_code": "String",
    "locality": "String (Ej: Madrid)"
  },
  "metadata": {
    "source": "String (Registro Administrativo)",
    "processed_at": "ISODate"
  }
}"""
add_json_block(doc, json_hotel, "A. Colección: accommodations_clean (Directorio de Alojamientos)")

# JSON B: Museos
json_museos = """{
  "_id": "ObjectId('...')",
  "name": "String (Ej: Museo del Prado)",
  "category": "String (Fijo: 'Museo')",
  "contact": "String (URL o Teléfono)",
  "location": { // <--- PATRÓN GEOESPACIAL (GeoJSON Standard)
    "type": "Point",
    "coordinates": [
      Double, // Longitud
      Double  // Latitud
    ]
  }
}"""
add_json_block(doc, json_museos, "B. Colección: cultural_pois_clean (Puntos de Interés Cultural)")

# JSON C: Eventos
json_eventos = """{
  "_id": "ObjectId('...')",
  "title": "String (Ej: Exposición Velázquez)",
  "category": "String (Ej: Evento, Teatro)",
  "date_info": { // <--- PATRÓN EMBEBIDO (Bucket Temporal Simple)
    "start": "String/ISODate (Fecha inicio)",
    "end": "String/ISODate (Fecha fin)"
  },
  "location": { // <--- PATRÓN GEOESPACIAL
    "type": "Point",
    "coordinates": [ Double, Double ]
  }
}"""
add_json_block(doc, json_eventos, "C. Colección: tourist_events_clean (Agenda de Actividades)")

# --- SECCIÓN 2: EXPLICACIÓN ---
doc.add_heading('2. Explicación Clara del Esquema', level=1)

p = doc.add_paragraph("El diseño sigue las mejores prácticas de modelado en MongoDB:")
p.style = 'List Bullet'
p = doc.add_paragraph()
runner = p.add_run("Modelo de Datos Embebido (Embedded Data Model):")
runner.bold = True
p.add_run(" Se agrupan detalles de dirección y fechas dentro del documento principal para evitar JOINs y mejorar la velocidad de lectura.")
p.style = 'List Bullet'

p = doc.add_paragraph()
runner = p.add_run("Patrón Geoespacial (Spatial Pattern):")
runner.bold = True
p.add_run(" Se utiliza estrictamente el formato GeoJSON (type: Point) en Museos y Eventos para habilitar índices '2dsphere' y consultas de proximidad.")
p.style = 'List Bullet'

p = doc.add_paragraph()
runner = p.add_run("Relaciones Implícitas:")
runner.bold = True
p.add_run(" La relación entre Museos y Eventos no es por ID, sino espacial (proximidad geográfica calculada en tiempo real).")
p.style = 'List Bullet'

# --- SECCIÓN 3: JUSTIFICACIÓN ---
doc.add_heading('3. Justificación del Diseño (Texto para Memoria)', level=1)

justificacion = (
    "Propuesta de Arquitectura de Datos NoSQL para la Plataforma Turística\n\n"
    "Para el desarrollo del sistema de análisis turístico, se ha implementado un esquema de base de datos "
    "orientado a documentos (MongoDB). Esta elección responde a la naturaleza heterogénea y no estructurada "
    "de las fuentes de datos Open Data.\n\n"
    "Estrategia de Modelado:\n"
    "El diseño del esquema se rige por un patrón desnormalizado y jerárquico. A diferencia de los modelos relacionales "
    "tradicionales (RDBMS) que priorizan la integridad referencial, este diseño prioriza la velocidad de lectura y la "
    "cohesión de la información.\n\n"
    "1. Patrón de Documentos Embebidos (Embedded Pattern): Se ha optado por embeber los detalles de dirección (address), "
    "información temporal (date_info) y geolocalización (location) dentro del documento raíz. Esto reduce drásticamente la latencia, "
    "eliminando la necesidad de uniones costosas.\n\n"
    "2. Implementación GeoJSON: Las colecciones 'cultural_pois_clean' y 'tourist_events_clean' implementan el estándar GeoJSON. "
    "Esta estructura permite la creación de índices geoespaciales '2dsphere', fundamentales para la personalización de la experiencia "
    "turística, permitiendo consultas complejas de proximidad ($near, $geoWithin) en tiempo real.\n\n"
    "3. Adaptabilidad (Schema Flexibility): Dado que el dataset de alojamientos carecía de coordenadas en origen, el esquema NoSQL "
    "permitió ingestar estos datos como directorio textual sin romper la integridad de las colecciones geolocalizadas."
)

p_just = doc.add_paragraph(justificacion)
p_just.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY

# Guardar
nombre_archivo = 'Diseno_Esquema_NoSQL.docx'
doc.save(nombre_archivo)
print(f"✅ Archivo '{nombre_archivo}' generado exitosamente.")

✅ Archivo 'Diseno_Esquema_NoSQL.docx' generado exitosamente.
